In [ ]:
#1
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

input_files = []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)
        input_files.append(path)
        print(path)

if not input_files:
    print("WARNING: /kaggle/input is empty.")
    print("Add competition data in Kaggle: Notebook -> Add Input -> Competitions")
    print("-> Walmart Recruiting - Store Sales Forecasting -> Add")
else:
    print(f"\nFound {len(input_files)} input file(s).")

In [ ]:
#2
!pip install prophet dagshub mlflow cmdstanpy -q

import warnings, logging
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from prophet import Prophet
import mlflow
import mlflow.pyfunc
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
logging.getLogger('prophet').setLevel(logging.ERROR)

dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
mlflow.set_experiment('Prophet_Training')
print('Ready.')

In [ ]:
#3
import os

COMPETITION_SLUG = "walmart-recruiting-store-sales-forecasting"

def find_competition_data_dir():
    preferred = f"/kaggle/input/competitions/{COMPETITION_SLUG}"
    if os.path.isdir(preferred):
        return preferred

    for root, _, files in os.walk("/kaggle/input"):
        names = set(files)
        if {"train.csv", "train.csv.zip"} & names:
            return root

    raise FileNotFoundError(
        "Competition data not found. In Kaggle open: Add Input -> Competitions -> "
        "Walmart Recruiting - Store Sales Forecasting -> Add, then re-run this notebook."
    )

def read_competition_csv(data_dir, stem):
    for name in (f"{stem}.csv", f"{stem}.csv.zip"):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f"Missing {stem}.csv or {stem}.csv.zip in {data_dir}")

DATA_DIR = find_competition_data_dir()
print(f"Using data directory: {DATA_DIR}")
print("Files:", sorted(os.listdir(DATA_DIR)))

train    = read_competition_csv(DATA_DIR, "train")
test     = read_competition_csv(DATA_DIR, "test")
stores   = read_competition_csv(DATA_DIR, "stores")
features = read_competition_csv(DATA_DIR, "features")

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])
features['Date'] = pd.to_datetime(features['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store','Date'], how='left')

test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store','Date'], how='left')

md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols]  = test[md_cols].fillna(0)

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

train = train.sort_values(['Store','Dept','Date']).reset_index(drop=True)
test  = test.sort_values(['Store','Dept','Date']).reset_index(drop=True)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Unique series: {train.groupby(['Store','Dept']).ngroups}")

## Prophet setup

Prophet expects columns `ds` (date) and `y` (target). It decomposes trend + yearly/weekly seasonality and supports holiday + regressor effects.
For this competition we keep the same cleaning as ARIMA/SARIMA and do not use lag features.

In [ ]:
#4
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

SPLIT_DATE = train['Date'].max() - pd.Timedelta(weeks=8)

train_set = train[train['Date'] <= SPLIT_DATE].copy()
val_set   = train[train['Date'] >  SPLIT_DATE].copy()

print(f"Train period: {train_set['Date'].min().date()} -> {train_set['Date'].max().date()}")
print(f"Val period  : {val_set['Date'].min().date()}   -> {val_set['Date'].max().date()}")
print(f"Val weeks   : {val_set['Date'].nunique()}")

In [ ]:
#5
def to_prophet_df(df, store, dept):
    part = df[(df['Store'] == store) & (df['Dept'] == dept)].copy()
    out = part.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    return out[['ds', 'y', 'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']]

store, dept = 1, 1
train_prophet = to_prophet_df(train_set, store, dept)
val_prophet = to_prophet_df(val_set, store, dept)

print(train_prophet.head(3))
print(f'Train weeks: {len(train_prophet)} | Val weeks: {len(val_prophet)}')

with mlflow.start_run(run_name='Prophet_Cleaning'):
    mlflow.log_params({
        'markdown_imputation': 'fill_zero',
        'negative_sales': 'clip_to_zero',
        'validation_weeks': 8,
        'model_input_format': 'ds_y',
        'note': 'no lag/rolling features for Prophet',
    })
    mlflow.log_text(
        f'Train weeks: {len(train_prophet)}\nVal weeks: {len(val_prophet)}\nStore: {store} Dept: {dept}\n',
        'data_summary.txt',
    )
    print('Cleaning run logged')

In [ ]:
#6
def fit_predict_prophet(train_df, val_df, params, regressors=None, use_holidays=False):
    m = Prophet(**params)
    if use_holidays:
        m.add_country_holidays(country_name='US')
    for reg in regressors or []:
        m.add_regressor(reg)
    fit_cols = ['ds', 'y'] + (regressors or [])
    m.fit(train_df[fit_cols])

    future = val_df[['ds'] + (regressors or [])].copy()
    forecast = m.predict(future)
    preds = np.clip(forecast['yhat'].values, 0, None)
    score = wmae(val_df['y'].values, preds, val_df['IsHoliday'].values)
    return m, preds, score

BASE_PARAMS = {
    'yearly_seasonality': True,
    'weekly_seasonality': True,
    'daily_seasonality': False,
    'seasonality_mode': 'additive',
}

m_base, preds_base, score_base = fit_predict_prophet(
    train_prophet, val_prophet, BASE_PARAMS, regressors=None, use_holidays=False
)
print(f'Prophet baseline WMAE (Store {store}, Dept {dept}): {score_base:,.2f}')

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.plot(train_prophet['ds'], train_prophet['y'], color='#58a6ff', linewidth=1.5, label='Train')
ax.plot(val_prophet['ds'], val_prophet['y'], color='#3fb950', linewidth=1.5, label='Actual val')
ax.plot(val_prophet['ds'], preds_base, color='#f78166', linewidth=1.5, linestyle='--', label='Prophet baseline')
ax.axvline(SPLIT_DATE, color='white', alpha=0.4, linestyle=':')
ax.set_title(f'Prophet baseline forecast WMAE={score_base:,.0f}', color='white')
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.savefig('prophet_baseline_forecast.png', dpi=130, facecolor='#0f1117')
plt.show()

with mlflow.start_run(run_name='Prophet_Baseline'):
    mlflow.log_params(BASE_PARAMS)
    mlflow.log_metric('val_wmae', score_base)
    mlflow.log_artifact('prophet_baseline_forecast.png', 'plots')

In [ ]:
#7
m_hol, preds_hol, score_hol = fit_predict_prophet(
    train_prophet, val_prophet, BASE_PARAMS, regressors=None, use_holidays=True
)
print(f'Prophet + US holidays WMAE: {score_hol:,.2f}')

with mlflow.start_run(run_name='Prophet_Holidays'):
    mlflow.log_params({**BASE_PARAMS, 'country_holidays': 'US'})
    mlflow.log_metric('val_wmae', score_hol)

In [ ]:
#8
REGRESSOR_SETS = {
    'Prophet_Regressor_IsHoliday': ['IsHoliday'],
    'Prophet_Regressor_Economic': ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
    'Prophet_Regressor_Full': ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
}

regressor_results = {'Prophet_Baseline': score_base, 'Prophet_Holidays': score_hol}

for run_name, regs in REGRESSOR_SETS.items():
    _, preds, score = fit_predict_prophet(
        train_prophet, val_prophet, BASE_PARAMS, regressors=regs, use_holidays=True
    )
    regressor_results[run_name] = score
    print(f'{run_name}: WMAE={score:,.2f}')
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({**BASE_PARAMS, 'regressors': regs, 'country_holidays': 'US'})
        mlflow.log_metric('val_wmae', score)

In [ ]:
#9
HYPERPARAM_GRID = {
    'Prophet_HP_FlexibleTrend': {
        'changepoint_prior_scale': 0.5,
        'seasonality_prior_scale': 10.0,
        'holidays_prior_scale': 10.0,
    },
    'Prophet_HP_Conservative': {
        'changepoint_prior_scale': 0.05,
        'seasonality_prior_scale': 1.0,
        'holidays_prior_scale': 1.0,
    },
    'Prophet_HP_Multiplicative': {
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.1,
        'seasonality_prior_scale': 5.0,
    },
}

best_regs = min(REGRESSOR_SETS, key=lambda k: regressor_results[k])
best_regressors = REGRESSOR_SETS[best_regs]
hyper_results = {}

for run_name, extra_params in HYPERPARAM_GRID.items():
    params = {**BASE_PARAMS, **extra_params}
    _, _, score = fit_predict_prophet(
        train_prophet, val_prophet, params, regressors=best_regressors, use_holidays=True
    )
    hyper_results[run_name] = score
    print(f'{run_name}: WMAE={score:,.2f}')
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({**params, 'regressors': best_regressors, 'country_holidays': 'US'})
        mlflow.log_metric('val_wmae', score)

In [ ]:
#10
def evaluate_prophet_subset(params, regressors, use_holidays=True):
    np.random.seed(42)
    all_pairs = train.groupby(['Store', 'Dept']).size()
    valid_pairs = all_pairs[all_pairs >= 80].index.tolist()
    sample_pairs = [valid_pairs[i] for i in np.random.choice(len(valid_pairs), size=50, replace=False)]

    preds_all, n_failed = [], 0
    for store_i, dept_i in sample_pairs:
        tr = to_prophet_df(train_set, store_i, dept_i)
        vl = to_prophet_df(val_set, store_i, dept_i)
        if len(tr) < 80 or len(vl) == 0:
            n_failed += 1
            continue
        try:
            _, preds, _ = fit_predict_prophet(tr, vl, params, regressors=regressors, use_holidays=use_holidays)
            for idx, (_, row) in enumerate(vl.iterrows()):
                if idx < len(preds):
                    preds_all.append({
                        'Actual': row['y'],
                        'Predicted': preds[idx],
                        'IsHoliday': row['IsHoliday'],
                    })
        except Exception:
            n_failed += 1

    if not preds_all:
        return None, n_failed
    pred_df = pd.DataFrame(preds_all)
    score = wmae(pred_df['Actual'].values, pred_df['Predicted'].values, pred_df['IsHoliday'].values)
    return score, n_failed

single_series_candidates = {**regressor_results, **hyper_results}
best_single_name = min(single_series_candidates, key=single_series_candidates.get)
print(f'Best single-series config: {best_single_name} -> {single_series_candidates[best_single_name]:,.2f}')

if best_single_name in HYPERPARAM_GRID:
    champion_params = {**BASE_PARAMS, **HYPERPARAM_GRID[best_single_name]}
    champion_regressors = best_regressors
elif best_single_name in REGRESSOR_SETS:
    champion_params = BASE_PARAMS.copy()
    champion_regressors = REGRESSOR_SETS[best_single_name]
elif best_single_name == 'Prophet_Holidays':
    champion_params = BASE_PARAMS.copy()
    champion_regressors = []
else:
    champion_params = BASE_PARAMS.copy()
    champion_regressors = []

subset_score, subset_failed = evaluate_prophet_subset(
    champion_params, champion_regressors, use_holidays=True
)
print(f'Subset50 WMAE: {subset_score:,.2f} (failed series: {subset_failed})')

with mlflow.start_run(run_name='Prophet_Subset50'):
    mlflow.log_params({**champion_params, 'regressors': champion_regressors, 'country_holidays': 'US'})
    mlflow.log_metric('val_wmae', subset_score)
    mlflow.log_metric('n_failed', subset_failed)
    mlflow.log_param('source_config', best_single_name)

In [ ]:
#11
all_scores = {**single_series_candidates, 'Prophet_Subset50': subset_score}
all_scores = {k: v for k, v in all_scores.items() if v is not None}

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
names = list(all_scores.keys())
scores = list(all_scores.values())
colors = ['#8b949e'] * len(names)
colors[int(np.argmin(scores))] = '#f78166'
bars = ax.bar(names, scores, color=colors, edgecolor='none', width=0.5)
ax.bar_label(bars, labels=[f'{v:,.1f}' for v in scores], padding=5, color='white', fontsize=9)
ax.set_title('Prophet runs comparison (WMAE, lower is better)', color='white', fontsize=13)
ax.tick_params(colors='#8b949e', axis='x', rotation=25)
plt.tight_layout()
plt.savefig('prophet_all_runs.png', dpi=130, facecolor='#0f1117')
plt.show()

champion_model, champion_preds, champion_val_score = fit_predict_prophet(
    train_prophet,
    val_prophet,
    champion_params,
    regressors=champion_regressors,
    use_holidays=True,
)

class ProphetPreprocessor(BaseEstimator):
    def __init__(self, regressors=None):
        self.regressors = regressors or []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        out = X.copy()
        if 'Date' in out.columns:
            out = out.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
        return out

class ProphetRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, prophet_model, regressors=None):
        self.prophet_model = prophet_model
        self.regressors = regressors or []
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        cols = ['ds'] + self.regressors
        forecast = self.prophet_model.predict(X[cols])
        return np.clip(forecast['yhat'].values, 0, None)

pipeline = Pipeline([
    ('preprocess', ProphetPreprocessor(regressors=champion_regressors)),
    ('model', ProphetRegressor(champion_model, regressors=champion_regressors)),
])

class ProphetPyFuncModel(mlflow.pyfunc.PythonModel):
    def __init__(self, prophet_model, regressors=None):
        self.prophet_model = prophet_model
        self.regressors = regressors or []

    def predict(self, context, model_input):
        cols = ['ds'] + self.regressors
        forecast = self.prophet_model.predict(model_input[cols])
        return np.clip(forecast['yhat'].values, 0, None)

with mlflow.start_run(run_name='Prophet_Champion'):
    mlflow.log_params({**champion_params, 'regressors': champion_regressors, 'country_holidays': 'US'})
    mlflow.log_param('best_config', best_single_name)
    mlflow.log_metric('val_wmae_single_series', champion_val_score)
    mlflow.log_metric('val_wmae_subset50', subset_score)
    mlflow.log_artifact('prophet_all_runs.png', 'plots')

    mlflow.pyfunc.log_model(
        python_model=ProphetPyFuncModel(champion_model, champion_regressors),
        artifact_path='prophet-model',
        registered_model_name='Prophet_Walmart_Forecaster',
    )
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path='prophet-pipeline',
        registered_model_name='Prophet_Walmart_Pipeline',
    )
    print('Champion Prophet model and pipeline registered in MLflow Model Registry')